# RunPod Workers!

Prior to running this file, a RUNPOD API KEY is needed.
An ssh key is also needed. See the runpod-ssh-how-to.docx file in this same directory for instructions and an example on how to do so.

In [1]:
import os
import time
import runpod
import paramiko  # for SSH
from pathlib import Path

# ───────────────────────────────────────────────
# CONFIGURATION
# ───────────────────────────────────────────────
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]  # set in your environment
runpod.api_key = RUNPOD_API_KEY

In [2]:
import runpod
print(runpod.get_gpus())

[{'id': 'AMD Instinct MI300X OAM', 'displayName': 'MI300X', 'memoryInGb': 192}, {'id': 'NVIDIA A100 80GB PCIe', 'displayName': 'A100 PCIe', 'memoryInGb': 80}, {'id': 'NVIDIA A100-SXM4-80GB', 'displayName': 'A100 SXM', 'memoryInGb': 80}, {'id': 'NVIDIA A40', 'displayName': 'A40', 'memoryInGb': 48}, {'id': 'NVIDIA B200', 'displayName': 'B200', 'memoryInGb': 180}, {'id': 'NVIDIA GeForce RTX 3070', 'displayName': 'RTX 3070', 'memoryInGb': 8}, {'id': 'NVIDIA GeForce RTX 3080', 'displayName': 'RTX 3080', 'memoryInGb': 10}, {'id': 'NVIDIA GeForce RTX 3080 Ti', 'displayName': 'RTX 3080 Ti', 'memoryInGb': 12}, {'id': 'NVIDIA GeForce RTX 3090', 'displayName': 'RTX 3090', 'memoryInGb': 24}, {'id': 'NVIDIA GeForce RTX 4070 Ti', 'displayName': 'RTX 4070 Ti', 'memoryInGb': 12}, {'id': 'NVIDIA GeForce RTX 4080', 'displayName': 'RTX 4080', 'memoryInGb': 16}, {'id': 'NVIDIA GeForce RTX 4080 SUPER', 'displayName': 'RTX 4080 SUPER', 'memoryInGb': 16}, {'id': 'NVIDIA GeForce RTX 4090', 'displayName': 'RTX

## TODO: Intelligent GPU type selection based on model size using runpod.get_gpus() and the memoryInGb field

In [4]:
# Inteligent GPU type selection based on model size using runpod.get_gpus() and the memoryInGb field
# def select_gpu_type(model_name):
#     gpus = runpod.get_gpus()
#     gpu_map = {gpu['displayName']: gpu['id'] for gpu in gpus if gpu['isAvailable']}
    
#     if "70b" in model_name or "gemma-2-70b" in model_name:
#         return gpu_map.get("NVIDIA A100 80GB") or gpu_map.get("NVIDIA A100 40GB")
#     elif "13b" in model_name or "gemma-2-13b" in model_name:
#         return gpu_map.get("NVIDIA RTX A6000")
#     else:
#         return gpu_map.get("NVIDIA GeForce RTX 4090")

# Taking Inventory

## Seeing what is currently available

In [43]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime


def summarize_eval_results(base_output_dir: str) -> pd.DataFrame:
    base_path = Path(base_output_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"Directory not found: {base_output_dir}")

    rows = []

    for results_file in base_path.rglob("results_*.json"):
        try:
            with open(results_file, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"⚠️ Could not parse {results_file}: {e}")
            continue

        # ----- Safe access wrappers -----
        config = data.get("config") or {}                      # <-- FIX TO ACCOMADATE NON-RUNPOD RUNS.
        configs = data.get("configs") or {}
        results = data.get("results") or {}
        n_samples_dict = data.get("n-samples") or {}

        # ----- Identify task -----
        task_name = next(iter(results.keys()), "unknown")

        # ----- Identify model -----
        model_name = (
            data.get("model_name")
            or config.get("model_args")
            or "unknown"
        )
        folder_model_name = results_file.parent.name
        ollama_name = folder_model_name.replace("__", ":")

        # ----- Metrics -----
        task_metrics = results.get(task_name, {}) or {}
        metric_keys = list(task_metrics.keys())
        score = None
        metric_type = None

        for key in ["exact_match,strict-match", "exact_match,none", "exact_match"]:
            if key in task_metrics:
                try:
                    score = float(task_metrics[key])
                except Exception:
                    score = 0.0
                metric_type = key
                break

        if score is None:
            numeric = [v for v in task_metrics.values() if isinstance(v, (int, float))]
            score = float(numeric[0]) if numeric else 0.0
            metric_type = "unknown"

        # ----- n-samples -----
        nsamples = (
            n_samples_dict.get(task_name, {}).get("effective")
            if isinstance(n_samples_dict.get(task_name), dict)
            else 0
        )

        # ----- Generation params -----
        gen_kwargs = config.get("gen_kwargs") or {}

        temp = gen_kwargs.get("temperature")
        max_toks = (
            configs.get(task_name, {})
            .get("generation_kwargs", {})
            .get("max_tokens")
            or configs.get(task_name, {})
            .get("generation_kwargs", {})
            .get("max_gen_toks")
        )

        # ----- GPU -----
        gpu_info = data.get("pretty_env_info", "") or ""
        gpu_count = gpu_info.count("GPU ")

        # ----- Date -----
        date_epoch = data.get("date", 0)
        try:
            date_str = datetime.utcfromtimestamp(date_epoch).strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            date_str = "unknown"

        # ----- Append -----
        rows.append({
            "Task": task_name,
            "Model Folder": folder_model_name,
            "Ollama Name": ollama_name,
            "Metric Type": metric_type,
            "Score": round(score, 6),
            "n-Samples": nsamples,
            "Temperature": temp,
            "Max Tokens": max_toks,
            "GPU Count": gpu_count,
            "Eval Date": date_str,
            "Result File": results_file.name,
            "Result Path": str(results_file)
        })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    # Sorting
    df.sort_values(["Task", "Model Folder", "Eval Date"], inplace=True)

    # Best-per-model selection with metric priority
    ordering = {"exact_match,strict-match": 0, "exact_match,none": 1, "unknown": 2}
    best_df = (
        df.sort_values(
            ["Metric Type"],
            key=lambda s: s.apply(lambda x: ordering.get(x, 3))
        )
        .groupby(["Task", "Model Folder"], as_index=False)
        .first()
    )

    return best_df


In [44]:
if __name__ == "__main__":
    # Example usage:
    base_output_dir = "../phase4_inference/output"
    df = summarize_eval_results(base_output_dir)
    print(df.to_string(index=False))
    # Optionally save
    # df.to_csv("evaluation_summary.csv", index=False)

C:\Users\rabel\AppData\Local\Temp\ipykernel_25404\3470769258.py:87: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  date_str = datetime.utcfromtimestamp(date_epoch).strftime("%Y-%m-%d %H:%M:%S")


           Task                 Model Folder                 Ollama Name              Metric Type    Score  n-Samples  Temperature  Max Tokens  GPU Count           Eval Date                             Result File                                                                                                     Result Path
    sysengbench anthropic__claude-sonnet-4.5 anthropic:claude-sonnet-4.5 exact_match,strict-match 0.948427       1144          NaN          10          1 2025-11-16 09:11:03 results_2025-11-16T04-31-27.456510.json     ..\phase4_inference\output\sysengbench\anthropic__claude-sonnet-4.5\results_2025-11-16T04-31-27.456510.json
    sysengbench                devstral__24b                devstral:24b exact_match,strict-match 0.930070       1144          0.0          10          2 2025-11-19 22:08:25 results_2025-11-19T22-11-13.670981.json                    ..\phase4_inference\output\sysengbench\devstral__24b\results_2025-11-19T22-11-13.670981.json
    sysengbench       

## status dataframe

In [45]:
import json
import pandas as pd
from pathlib import Path

def generate_model_task_matrix_with_scores_and_samples(base_output_dir: str) -> pd.DataFrame:
    """
    Build a matrix with:
      * 'Model Folder Name' (safe for filesystem)
      * 'Model Ollama Name' (with ':' restored)
      * One column per task showing ✅/❌ with counts, best score, and sample count.

    Directory structure expected:
        base_output_dir/task_name/model_folder_name/
            results_<timestamp>.json
            samples_<task>_<timestamp>.jsonl
    """
    base_path = Path(base_output_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"Directory does not exist: {base_output_dir}")

    # Discover all tasks and all folder names
    task_names = [p.name for p in base_path.iterdir() if p.is_dir()]
    model_folder_names = set()
    for task in task_names:
        for model_dir in (base_path / task).iterdir():
            if model_dir.is_dir():
                model_folder_names.add(model_dir.name)

    # Prepare matrix with two leading columns
    matrix = pd.DataFrame(index=sorted(model_folder_names),
                          columns=["Model Folder Name", "Model Ollama Name"] + sorted(task_names))

    for model_folder in sorted(model_folder_names):
        # Fill in the two identifying columns
        matrix.at[model_folder, "Model Folder Name"] = model_folder
        # Convert "__" back to ":" for correct Ollama name
        matrix.at[model_folder, "Model Ollama Name"] = model_folder.replace("__", ":")

        for task in task_names:
            model_dir = base_path / task / model_folder
            if not model_dir.exists():
                matrix.at[model_folder, task] = "❌ (0 – max:0.0 – samples:0)"
                continue

            result_files = [f for f in model_dir.iterdir()
                            if f.name.startswith("results_") and f.suffix == ".json"]

            if not result_files:
                matrix.at[model_folder, task] = "❌ (0 – max:0.0 – samples:0)"
                continue

            best_score = 0.0
            best_result_file = None

            # Find highest scoring results file
            for rf in result_files:
                try:
                    with open(rf, "r") as fh:
                        data = json.load(fh)
                    if "results" in data and task in data["results"]:
                        task_data = data["results"][task]
                        score = (task_data.get("exact_match,strict-match")
                                 or task_data.get("exact_match")
                                 or max((v for v in task_data.values() if isinstance(v,(int,float))), default=0.0))
                        if score and score > best_score:
                            best_score = float(score)
                            best_result_file = rf
                except Exception as e:
                    print(f"Warning: could not parse {rf}: {e}")

            n_results = len(result_files)
            samples_exist = False
            max_samples_count = 0

            if best_result_file:
                ts = best_result_file.stem.replace("results_", "")
                expected_samples_prefix = f"samples_{task}_{ts}"
                for f in model_dir.iterdir():
                    if f.name.startswith(expected_samples_prefix) and f.suffix == ".jsonl":
                        samples_exist = True
                        with open(f, "r", encoding="utf-8") as sf:
                            count = sum(1 for _ in sf)
                        max_samples_count = count
                        break

            if best_result_file and samples_exist:
                matrix.at[model_folder, task] = f"✅ ({n_results}) – max:{best_score:.3f} – samples:{max_samples_count}"
            else:
                matrix.at[model_folder, task] = f"❌ ({n_results}) – max:{best_score:.3f} – samples:{max_samples_count}"

    matrix.index.name = "Model Folder Name (index)"
    return matrix


If a task isn't showing up, simply make sure a folder is present in the downloaded_output directory and it will add the column for that task.

In [46]:

base_output_dir = Path("output")
print(f"Base output dir: {base_output_dir}")

# Generate and prepare matrix
print("Generating matrix...")
matrix_df = generate_model_task_matrix_with_scores_and_samples(str(base_output_dir))

matrix_df = matrix_df.reset_index(drop=True)

pd.set_option('display.expand_frame_repr', False)  # don’t wrap whole rows
pd.set_option('display.max_colwidth', None)        # don’t truncate/wrap cells

# View in Jupyter
display(matrix_df)

# Optional export
# matrix_df.to_csv("model_task_matrix_with_scores_and_samples.csv")


Base output dir: output
Generating matrix...


,Model Folder Name,Model Ollama Name,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d,sysengbench-osq
0,anthropic__claude-sonnet-4.5,anthropic:claude-sonnet-4.5,✅ (1) – max:0.948 – samples:1144,✅ (1) – max:0.940 – samples:1144,✅ (1) – max:0.959 – samples:1144,✅ (1) – max:0.968 – samples:1144,✅ (1) – max:0.932 – samples:1144,✅ (1) – max:0.065 – samples:845
1,devstral__24b,devstral:24b,✅ (1) – max:0.930 – samples:1144,✅ (1) – max:0.898 – samples:1144,✅ (1) – max:0.944 – samples:1144,✅ (1) – max:0.941 – samples:1144,✅ (1) – max:0.908 – samples:1144,✅ (1) – max:0.022 – samples:845
2,gemma3__12b,gemma3:12b,✅ (1) – max:0.891 – samples:1144,✅ (1) – max:0.863 – samples:1144,✅ (1) – max:0.893 – samples:1144,✅ (1) – max:0.907 – samples:1144,✅ (1) – max:0.889 – samples:1144,✅ (1) – max:0.017 – samples:845
3,gemma3__27b,gemma3:27b,✅ (1) – max:0.922 – samples:1144,✅ (1) – max:0.902 – samples:1144,✅ (1) – max:0.917 – samples:1144,✅ (1) – max:0.935 – samples:1144,✅ (1) – max:0.912 – samples:1144,❌ (1) – max:0.000 – samples:0
4,google__gemini-2.5-flash,google:gemini-2.5-flash,✅ (1) – max:0.949 – samples:1144,✅ (1) – max:0.954 – samples:1144,✅ (1) – max:0.945 – samples:1144,✅ (1) – max:0.949 – samples:1144,✅ (1) – max:0.946 – samples:1144,✅ (1) – max:0.063 – samples:845
5,llama3.2__3b,llama3.2:3b,✅ (1) – max:0.832 – samples:1144,✅ (1) – max:0.691 – samples:1144,✅ (1) – max:0.813 – samples:1144,✅ (1) – max:0.917 – samples:1144,✅ (1) – max:0.728 – samples:1144,❌ (1) – max:0.000 – samples:0
6,llama3.3__70b,llama3.3:70b,✅ (1) – max:0.929 – samples:1144,✅ (1) – max:0.925 – samples:1144,✅ (1) – max:0.940 – samples:1144,✅ (1) – max:0.923 – samples:1144,✅ (1) – max:0.905 – samples:1144,✅ (1) – max:0.027 – samples:845
7,llama4__16x17b,llama4:16x17b,✅ (1) – max:0.938 – samples:1144,✅ (1) – max:0.926 – samples:1144,✅ (1) – max:0.948 – samples:1144,✅ (1) – max:0.924 – samples:1144,✅ (1) – max:0.923 – samples:1144,✅ (1) – max:0.015 – samples:845
8,mistral-large__123b,mistral-large:123b,✅ (1) – max:0.882 – samples:1144,✅ (1) – max:0.893 – samples:1144,✅ (1) – max:0.878 – samples:1144,✅ (1) – max:0.884 – samples:1144,✅ (1) – max:0.835 – samples:1144,✅ (1) – max:0.040 – samples:845
9,mistral-small3.2__24b,mistral-small3.2:24b,✅ (1) – max:0.944 – samples:1144,✅ (1) – max:0.899 – samples:1144,✅ (1) – max:0.938 – samples:1144,✅ (1) – max:0.948 – samples:1144,✅ (1) – max:0.915 – samples:1144,✅ (1) – max:0.044 – samples:845


## Adding other models manually to status dataframe

In [27]:
# Optionally Adding in new models/tasks
def register_model(
    matrix_df: pd.DataFrame,
    ollama_name: str,
    task_overrides: dict | None = None,
    folder_name: str | None = None
) -> pd.DataFrame:
    """
    Register a new model into the evaluation matrix using only the Ollama name.

    Parameters
    ----------
    matrix_df : pd.DataFrame
        Existing evaluation matrix.
    ollama_name : str
        The Ollama model identifier, e.g., "llama3.1:70b".
    task_overrides : dict, optional
        Dict mapping task names -> status strings (e.g. {"sysengbench-c": "✅ samples: 2000"})
        Any task not included defaults to "❌".
    folder_name : str, optional
        Optional folder name; defaults to ollama_name with ':' replaced by '_'.

    Returns
    -------
    Updated matrix_df with the new model appended.
    """

    # Derive folder name if not supplied
    folder = folder_name or ollama_name.replace(":", "_")

    # Identify existing task columns
    task_columns = [
        c for c in matrix_df.columns
        if c not in ("Model Folder Name", "Model Ollama Name")
    ]

    # Initialize new row
    row = {
        "Model Folder Name": folder,
        "Model Ollama Name": ollama_name,
    }

    # Fill each task with overrides or default ❌
    for task in task_columns:
        if task_overrides and task in task_overrides:
            row[task] = task_overrides[task]
        else:
            row[task] = "❌"

    # Append row
    return pd.concat([matrix_df, pd.DataFrame([row])], ignore_index=True)



In [ ]:
# IF I END UP HAVING TO RE-RUN MODELS, ADD THEM HERE:

# # only flagship, mostly frontier models:
# First Round: non-thinking, core models
# matrix_df = register_model(matrix_df, "llama3.3:70b") 
# matrix_df = register_model(matrix_df, "llama4:16x17b") # "Scout"
# matrix_df = register_model(matrix_df, "phi4:14b")   
# matrix_df = register_model(matrix_df, "gemma3:27b")
# matrix_df = register_model(matrix_df, "mistral-large:123b") # 73GB

# Second Round
# matrix_df = register_model(matrix_df, "mistral-small3.2:24b") 
# matrix_df = register_model(matrix_df, "llama3.2:3b")
# matrix_df = register_model(matrix_df, "phi4-mini:3.8b")       
# matrix_df = register_model(matrix_df, "gemma3:12b")
# matrix_df = register_model(matrix_df, "devstral:24b")

# Third Round
matrix_df = register_model(matrix_df, "llama3.2:1b")  
matrix_df = register_model(matrix_df, "gemma3:1b")              
matrix_df = register_model(matrix_df, "gemma3:4b")
matrix_df = register_model(matrix_df, "mixtral:8x22b")
matrix_df = register_model(matrix_df, "phi3:14b")
matrix_df = register_model(matrix_df, "phi3.5:3.8b")


# # smaller models or non-flagship
# matrix_df = register_model(matrix_df, "deepseek-r1:8b")
# matrix_df = register_model(matrix_df, "deepseek-r1:14b")
# matrix_df = register_model(matrix_df, "gpt-oss:20b")                        

# matrix_df = register_model(matrix_df, "phi4-reasoning:14b")     
# matrix_df = register_model(matrix_df, "qwen3:0.6b")             
# matrix_df = register_model(matrix_df, "qwen3:1.7b")
# matrix_df = register_model(matrix_df, "qwen3:4b")
# matrix_df = register_model(matrix_df, "qwen3:8b")
# matrix_df = register_model(matrix_df, "qwen3:14b")
# matrix_df = register_model(matrix_df, "qwen3:30b")

# matrix_df = register_model(matrix_df, "phi3:3.8b")
# matrix_df = register_model(matrix_df, "phi3-medium")
# matrix_df = register_model(matrix_df, "phi3.5:3.8b")

# matrix_df = register_model(matrix_df, "gemma3:270m")
# matrix_df = register_model(matrix_df, "gemma3n:e2b")            
# matrix_df = register_model(matrix_df, "gemma3n:e4b")            
         


# matrix_df = register_model(matrix_df, "mistral:7b")

# # thinking
# matrix_df = register_model(matrix_df, "deepseek-r1:70b") # thinking 
# matrix_df = register_model(matrix_df, "deepseek-r1:32b") # thinking
# matrix_df = register_model(matrix_df, "gpt-oss:120b")   # thinking      
# matrix_df = register_model(matrix_df, "magistral:24b") # thinking
# matrix_df = register_model(matrix_df, "qwen3:32b") # thinking             


TO DO: (Optionally) Import a model list from a csv

In [29]:
display(matrix_df)

,Model Folder Name,Model Ollama Name,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d,sysengbench-osq
0,anthropic__claude-sonnet-4.5,anthropic:claude-sonnet-4.5,✅ (1) – max:0.948 – samples:1144,✅ (1) – max:0.940 – samples:1144,✅ (1) – max:0.959 – samples:1144,✅ (1) – max:0.968 – samples:1144,✅ (1) – max:0.932 – samples:1144,✅ (1) – max:0.065 – samples:845
1,gemma3__27b,gemma3:27b,✅ (1) – max:0.922 – samples:1144,✅ (1) – max:0.902 – samples:1144,✅ (1) – max:0.917 – samples:1144,✅ (1) – max:0.935 – samples:1144,✅ (1) – max:0.912 – samples:1144,❌ (1) – max:0.000 – samples:0
2,google__gemini-2.5-flash,google:gemini-2.5-flash,✅ (1) – max:0.949 – samples:1144,✅ (1) – max:0.954 – samples:1144,✅ (1) – max:0.945 – samples:1144,✅ (1) – max:0.949 – samples:1144,✅ (1) – max:0.946 – samples:1144,✅ (1) – max:0.063 – samples:845
3,llama3.3__70b,llama3.3:70b,✅ (1) – max:0.929 – samples:1144,✅ (1) – max:0.925 – samples:1144,✅ (1) – max:0.940 – samples:1144,✅ (1) – max:0.923 – samples:1144,✅ (1) – max:0.905 – samples:1144,✅ (1) – max:0.027 – samples:845
4,llama4__16x17b,llama4:16x17b,✅ (1) – max:0.938 – samples:1144,✅ (1) – max:0.926 – samples:1144,✅ (1) – max:0.948 – samples:1144,✅ (1) – max:0.924 – samples:1144,✅ (1) – max:0.923 – samples:1144,✅ (1) – max:0.015 – samples:845
5,mistral-large__123b,mistral-large:123b,✅ (1) – max:0.882 – samples:1144,✅ (1) – max:0.893 – samples:1144,✅ (1) – max:0.878 – samples:1144,✅ (1) – max:0.884 – samples:1144,✅ (1) – max:0.835 – samples:1144,✅ (1) – max:0.040 – samples:845
6,openai__gpt-4.1,openai:gpt-4.1,✅ (1) – max:0.959 – samples:1144,✅ (1) – max:0.963 – samples:1144,✅ (1) – max:0.951 – samples:1144,✅ (1) – max:0.961 – samples:1144,✅ (1) – max:0.944 – samples:1144,✅ (1) – max:0.078 – samples:845
7,phi4__14b,phi4:14b,✅ (1) – max:0.937 – samples:1144,✅ (1) – max:0.920 – samples:1144,✅ (1) – max:0.934 – samples:1144,✅ (1) – max:0.934 – samples:1144,✅ (1) – max:0.913 – samples:1144,✅ (1) – max:0.004 – samples:845
8,mistral-small3.2_24b,mistral-small3.2:24b,❌,❌,❌,❌,❌,❌
9,llama3.2_3b,llama3.2:3b,❌,❌,❌,❌,❌,❌


## Finalizing what to run

### Option 1: Any Xs in the status dataframe

In [30]:
def extract_missing_from_matrix(matrix_df: pd.DataFrame) -> dict:
    """
    From the new matrix (with Model Folder Name and Model Ollama Name columns),
    return {ollama_model_name: [missing tasks]}.

    Example:
        {
            "deepseek-r1:14b": ["sysengbench", "sysengbench-a"],
            "gemma3:27b": ["sysengbench-d"]
        }
    """
    missing = {}

    # Identify which columns are actual tasks
    # (everything except our name columns)
    task_columns = [c for c in matrix_df.columns
                    if c not in ("Model Folder Name", "Model Ollama Name")]

    for _, row in matrix_df.iterrows():
        ollama_name = row["Model Ollama Name"]
        missing_tasks = [
            task for task in task_columns
            if isinstance(row[task], str) and row[task].startswith("❌")
        ]
        if missing_tasks:
            missing[ollama_name] = missing_tasks

    return missing


In [31]:
# assuming you already created the matrix with scores and samples
# matrix_df = generate_model_task_matrix_with_scores_and_samples(base_output_dir)

print("Model-task pairs to run:")

missing_dict = extract_missing_from_matrix(matrix_df)

print(missing_dict)

# Count all model-task pairs
total_permutations = sum(len(tasks) for tasks in missing_dict.values())

print(f"Total model-task permutations: {total_permutations}")

Model-task pairs to run:
{'gemma3:27b': ['sysengbench-osq'], 'mistral-small3.2:24b': ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq'], 'llama3.2:3b': ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq'], 'phi4-mini:3.8b': ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq'], 'gemma3:12b': ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq'], 'devstral:24b': ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq']}
Total model-task permutations: 31


### Option 2: Any Xs in dataframe minus exemptions (e.g. a particular model or task)

In [32]:
import re
import pandas as pd

def extract_missing_from_matrix(matrix_df: pd.DataFrame) -> dict:
    """
    Return {ollama_model_name: [missing tasks]}, skipping known exemptions.

    Supports:
      1. Global model exemptions (skip entire model).
      2. Task-specific sample count exemptions.
      3. Task-specific model exemptions.
      4. Task-specific pattern exemptions.
    """

    # Exempt a model from ALL benchmark tasks
    GLOBAL_MODEL_EXEMPTIONS = [
        "gemma3:27b",     # example
        # # "phi4-reasoning:plus",   # add more here
        # "deepseek-r1:8b",
    ]

    EXEMPTIONS = {
        # "sysengbench-osq": {
        #     "samples": [845],
        #     "patterns": [r"\(1\)"]
        # },

        # "sysengbench-c": {
        #     "models": ["magistral:24b"]
        # },
    }

    missing = {}
    task_columns = [
        c for c in matrix_df.columns
        if c not in ("Model Folder Name", "Model Ollama Name")
    ]

    for _, row in matrix_df.iterrows():
        ollama_name = row["Model Ollama Name"]

        # 🔥 GLOBAL MODEL EXEMPTION — skip entire model
        if ollama_name in GLOBAL_MODEL_EXEMPTIONS:
            continue

        missing_tasks = []

        for task in task_columns:
            val = row[task]
            if not isinstance(val, str):
                continue

            exempted = False

            # Task-specific exemptions
            if task in EXEMPTIONS:
                rule = EXEMPTIONS[task]

                # Sample count-based exemption
                if "samples" in rule:
                    for exempt_count in rule["samples"]:
                        if re.search(rf"samples:\s*0*{exempt_count}\b", val):
                            exempted = True
                            break

                # Model-specific exemption at task level
                if not exempted and "models" in rule:
                    if ollama_name in rule["models"]:
                        exempted = True

                # Pattern-based exemption
                if not exempted and "patterns" in rule:
                    for pattern in rule["patterns"]:
                        if re.search(pattern, val):
                            exempted = True
                            break

            # The normal ❌ rule
            if not exempted and val.startswith("❌"):
                missing_tasks.append(task)

        if missing_tasks:
            missing[ollama_name] = missing_tasks

    return missing


In [33]:
# check for is the magistral exemption worked
# missing_dict = extract_missing_from_matrix(matrix_df)
# print(missing_dict.get("magistral:24b", []))


In [34]:
# assuming you already created the matrix with scores and samples
# matrix_df = generate_model_task_matrix_with_scores_and_samples(base_output_dir)

print("Model-task pairs to run:")

missing_dict = extract_missing_from_matrix(matrix_df)

print(missing_dict)

# Count all model-task pairs
total_permutations = sum(len(tasks) for tasks in missing_dict.values())

print(f"Total model-task permutations: {total_permutations}")

Model-task pairs to run:
{'mistral-small3.2:24b': ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq'], 'llama3.2:3b': ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq'], 'phi4-mini:3.8b': ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq'], 'gemma3:12b': ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq'], 'devstral:24b': ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq']}
Total model-task permutations: 30


# Running Multiple RunPod Containers

In [35]:
import os
import time
import runpod
import paramiko  # for SSH
from pathlib import Path

# ───────────────────────────────────────────────
# CONFIGURATION
# ───────────────────────────────────────────────
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]  # set in your environment
runpod.api_key = RUNPOD_API_KEY

## Read in the YAMLS

In [36]:
with open('sysengbench.yaml') as f:
    sysengbench_yaml = f.read()

with open('sysengbench-a.yaml') as f:
    sysengbench_a_yaml = f.read()

with open('sysengbench-b.yaml') as f:
    sysengbench_b_yaml = f.read()

with open('sysengbench-c.yaml') as f:
    sysengbench_c_yaml = f.read()

with open('sysengbench-d.yaml') as f:
    sysengbench_d_yaml = f.read()

with open('sysengbench-osq.yaml') as f:
    sysengbench_osq_yaml = f.read()

In [37]:
yaml_templates = {
    "sysengbench": sysengbench_yaml,
    "sysengbench-a": sysengbench_a_yaml,
    "sysengbench-b": sysengbench_b_yaml,
    "sysengbench-c": sysengbench_c_yaml,
    "sysengbench-d": sysengbench_d_yaml,
    "sysengbench-osq": sysengbench_osq_yaml,  # reuse base for osqa
}


## Option 1: One Model at a Time
Sequential: Checks for each step to be done before proceeding on.

In [ ]:
import os, time, stat, posixpath, paramiko, runpod
from pathlib import Path

def run_missing_models(missing_dict, image_name, gpu_type,
                       yaml_templates, base_output_dir, local_results_dir):
    """
    Loop over {model: [tasks]} and run each missing task in its own RunPod pod.
    Includes blocking waits, exit-code checks, and robust downloading.
    """
    ssh_key_path = os.path.expanduser("~/.ssh/id_ed25519")

    def run_and_check(ssh, cmd, desc):
        """Run a remote command and wait for completion, printing output and errors."""
        print(f"▶ {desc}: {cmd}")
        stdin, stdout, stderr = ssh.exec_command(cmd)
        exit_code = stdout.channel.recv_exit_status()
        out = stdout.read().decode()
        err = stderr.read().decode()
        if out: print(out)
        if err: print("stderr:", err)
        if exit_code != 0:
            raise RuntimeError(f"❌ Command failed [{desc}] with exit {exit_code}")
        print(f"✔ {desc} finished.")
        return out

    def download_dir(sftp, remote_dir, local_dir):
        """Safely download a remote directory tree if it exists."""
        try:
            entries = sftp.listdir_attr(remote_dir)
        except FileNotFoundError:
            print(f"⚠ No output directory found at {remote_dir}")
            return
        os.makedirs(local_dir, exist_ok=True)
        for entry in entries:
            remote_path = posixpath.join(remote_dir, entry.filename)
            local_path  = os.path.join(local_dir, entry.filename)
            if stat.S_ISDIR(entry.st_mode):
                download_dir(sftp, remote_path, local_path)
            else:
                sftp.get(remote_path, local_path)
                print(f"  ↓ {local_path}")

    for model, tasks in missing_dict.items():
        for task in tasks:
            print(f"\n=== Starting pod for {model} | task: {task} ===")
            pod_name = f"lm-eval-{model.replace(':','-')}-{task}-{int(time.time())}"
            pod = runpod.create_pod(
                name=pod_name,
                image_name=image_name,
                gpu_type_id=gpu_type,
                gpu_count=1,
                container_disk_in_gb=200,
                min_vcpu_count=4,
                min_memory_in_gb=16,
                ports="22/tcp,11434/http",
                env={"OLLAMA_HOST": "0.0.0.0", "PYTHONUNBUFFERED": "1"},
                support_public_ip=True,
                start_ssh=True
            )
            pod_id = pod["id"]
            print(f"Created pod: {pod_id}")

            # Wait for pod to be RUNNING and runtime ports available
            ssh_host = ssh_port = None
            while True:
                details = runpod.get_pod(pod_id)
                status = details.get("desiredStatus")
                print(f"  Current status: {status}")
                if status == "RUNNING":
                    runtime = details.get("runtime")
                    if runtime and runtime.get("ports"):
                        for p in runtime["ports"]:
                            if p["type"]=="tcp" and p["privatePort"]==22 and p["isIpPublic"]:
                                ssh_host, ssh_port = p["ip"], p["publicPort"]
                                break
                        if ssh_host:
                            break
                time.sleep(10)

            print(f"Pod running at {ssh_host}:{ssh_port}")

            ssh = paramiko.SSHClient()
            ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
            ssh.connect(ssh_host, port=ssh_port, username="root",
                        pkey=paramiko.Ed25519Key.from_private_key_file(ssh_key_path))
            print("SSH connected.")

            try:
                # Provision environment
                provisioning_cmds = [
                    ("Install lshw", "apt update && apt install -y lshw"),
                    ("Install Ollama", "curl -fsSL https://ollama.com/install.sh | sh"),
                    ("Start Ollama", "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &"),
                    ("Wait for Ollama", "sleep 10"),
                    ("Install lm_eval + Ollama Python", "pip install lm_eval lm_eval[api] ollama==0.3.3"),
                ]
                for desc, cmd in provisioning_cmds:
                    run_and_check(ssh, cmd, desc)

                # Pull the model
                run_and_check(ssh, f"ollama pull {model}", f"Pull model {model}")

                # Push task YAML
                yaml_path = f"/root/{task}.yaml"
                yaml_content = yaml_templates[task]
                push_cmd = f"cat > {yaml_path} <<'EOF'\n{yaml_content}\nEOF"
                run_and_check(ssh, push_cmd, f"Upload YAML for {task}")

                # Run lm_eval and wait until it exits
                output_dir = f"output/{task}/"
                lm_eval_cmd = f"""
                lm_eval \
                  --model local-chat-completions \
                  --model_args model='{model}',base_url='http://localhost:11434/v1/chat/completions',num_concurrent=1 \
                  --include_path ./ \
                  --tasks {task} \
                  --output {output_dir} \
                  --log_samples \
                  --num_fewshot 0 \
                  --batch_size auto \
                  --gen_kwargs temperature=0.0 \
                  --apply_chat_template
                """
                run_and_check(ssh, lm_eval_cmd, f"Run lm_eval for {task}")

                # Download results safely
                sftp = ssh.open_sftp()
                remote_dir = f"/root/{output_dir}"
                local_dir = os.path.join(local_results_dir, task)
                download_dir(sftp, remote_dir, local_dir)
                sftp.close()

            except Exception as e:
                print(f"❌ Error during run for {model} | task: {task}: {e}")

            finally:
                runpod.terminate_pod(pod_id)
                ssh.close()
                print(f"Pod {pod_id} terminated.")


In [45]:
run_missing_models(
    missing_dict=missing_dict,
    image_name=IMAGE_NAME,
    gpu_type=GPU_TYPE,
    yaml_templates=yaml_templates,
    base_output_dir="output",
    local_results_dir=local_results_dir
)



=== Starting pod for mistral:7b | task: sysengbench-c ===
raw_response: {'data': {'podFindAndDeployOnDemand': {'id': 'petfrcbjud7jay', 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'env': ['OLLAMA_HOST=0.0.0.0', 'PYTHONUNBUFFERED=1', 'PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'machineId': 'j65nym2f1yl5', 'machine': {'podHostId': 'petfrcbjud7jay-64411291'}}}}
Created pod: petfrcbjud7jay
  Current status: RUNNING
  Current status: RUNNING
  Current status: RUNNING
  Current status: RUNNING
Pod running at 194.68.245.44:22160
SSH connected.
▶ Install lshw: apt update && apt install -y lshw
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2008 kB]
Get:4 http://

## (PREFERRED) Option 2: Parallelized Models Running Concurrently

In [40]:
import os, time, stat, posixpath, paramiko, runpod
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

def run_single_model_task(model, task,
                          image_name, gpu_type,
                          yaml_templates,
                          base_output_dir,
                          local_results_dir,
                          log_dir):
    """
    Run exactly one model-task evaluation in its own RunPod pod.
    Returns (model, task, success_boolean).
    """
    log_file = Path(log_dir) / f"{model.replace(':','_')}__{task}.log"
    log_file.parent.mkdir(parents=True, exist_ok=True)

    def log(msg):
        stamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{stamp}] [{model} | {task}] {msg}"
        print(line)
        with open(log_file, "a", encoding="utf-8") as lf:
            lf.write(line + "\n")

    def run_and_check(ssh, cmd, desc):
        log(f"▶ {desc}")
        stdin, stdout, stderr = ssh.exec_command(cmd)
        exit_code = stdout.channel.recv_exit_status()
        out = stdout.read().decode()
        err = stderr.read().decode()
        if out: log(out)
        if err: log(f"stderr: {err}")
        if exit_code != 0:
            raise RuntimeError(f"{desc} failed with exit code {exit_code}")
        log(f"✔ {desc} finished.")

    def download_dir(sftp, remote_dir, local_dir):
        try:
            entries = sftp.listdir_attr(remote_dir)
        except FileNotFoundError:
            log(f"⚠ No output directory found at {remote_dir}")
            return
        os.makedirs(local_dir, exist_ok=True)
        for entry in entries:
            remote_path = posixpath.join(remote_dir, entry.filename)
            local_path  = os.path.join(local_dir, entry.filename)
            if stat.S_ISDIR(entry.st_mode):
                download_dir(sftp, remote_path, local_path)
            else:
                sftp.get(remote_path, local_path)
                log(f"↓ {local_path}")

    ssh_key_path = os.path.expanduser("~/.ssh/id_ed25519")

    try:
        # 1. Create pod
        log("Creating pod...")
        pod = runpod.create_pod(
            name=f"lm-eval-{model.replace(':','-')}-{task}-{int(time.time())}",
            image_name=image_name,
            gpu_type_id=gpu_type,
            gpu_count=1,  ########### Change for bigger models! ##################
            container_disk_in_gb=200,
            min_vcpu_count=4,
            min_memory_in_gb=16,
            ports="22/tcp,11434/http",
            env={"OLLAMA_HOST": "0.0.0.0", "PYTHONUNBUFFERED": "1"},
            support_public_ip=True,
            start_ssh=True
        )
        pod_id = pod["id"]
        log(f"Created pod: {pod_id}")

        # 2. Wait for pod to be RUNNING and runtime ports ready
        ssh_host = ssh_port = None
        while True:
            details = runpod.get_pod(pod_id)
            status = details.get("desiredStatus")
            log(f"Current status: {status}")
            if status == "RUNNING":
                runtime = details.get("runtime")
                if runtime and runtime.get("ports"):
                    for p in runtime["ports"]:
                        if p["type"]=="tcp" and p["privatePort"]==22 and p["isIpPublic"]:
                            ssh_host, ssh_port = p["ip"], p["publicPort"]
                            break
                    if ssh_host:
                        break
            time.sleep(10)
        log(f"Pod running at {ssh_host}:{ssh_port}")

        # 3. Connect via SSH
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(ssh_host, port=ssh_port, username="root",
                    pkey=paramiko.Ed25519Key.from_private_key_file(ssh_key_path))
        log("SSH connected.")

        # 4. Provision environment
        steps = [
            ("Install lshw", "apt update && apt install -y lshw"),
            ("Install Ollama", "curl -fsSL https://ollama.com/install.sh | sh"),
            ("Start Ollama", "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &"),
            ("Wait for Ollama", "sleep 10"),
            ("Install lm_eval + Ollama Python", "pip install lm_eval lm_eval[api] ollama==0.3.3"),
            (f"Pull model {model}", f"ollama pull {model}")
        ]
        for desc, cmd in steps:
            run_and_check(ssh, cmd, desc)

        # 5. Upload task YAML
        # yaml_path = f"/root/{task}.yaml"
        # yaml_path = f"/root/tasks/{task}.yaml"
        # yaml_content = yaml_templates[task]
        # run_and_check(ssh, f"cat > {yaml_path} <<'EOF'\n{yaml_content}\nEOF", f"Upload YAML for {task}")

        # 5. Upload task YAML

        # Ensure that the tasks directory exists inside the pod. Your base
        # image does not include /root/tasks by default, so we must create it.
        run_and_check(ssh, "mkdir -p /root/tasks", "Ensure tasks directory exists")

        # Path where the YAML file will be written inside the container.
        yaml_path = f"/root/tasks/{task}.yaml"

        # Retrieve the YAML content from the yaml_templates dictionary.
        # This should be a plain string — lm_eval tasks are typically provided
        # as static YAML strings.
        yaml_content = yaml_templates[task]

        # Construct the shell command that writes YAML content to a file using
        # a heredoc. The <<'EOF' form prevents variable expansion inside the YAML,
        # which keeps the content intact and avoids accidental substitutions.
        upload_cmd = f"cat > {yaml_path} <<'EOF'\n{yaml_content}\nEOF"

        # Run the command on the remote pod. If the exit code is not 0,
        # run_and_check() will raise an error and stop the pipeline.
        run_and_check(
            ssh,
            upload_cmd,
            f"Upload YAML for {task}"
        )

        # 6. Run lm_eval and wait
        # optional no-thinking approach:
        # --model_args model='{model}',base_url='http://localhost:11434/v1/chat/completions',num_concurrent=1,think=false \
        output_dir = f"output/{task}/"
        lm_eval_cmd = f"""
        lm_eval \
          --model local-chat-completions \
          --model_args model='{model}',base_url='http://localhost:11434/v1/chat/completions',num_concurrent=1 \
          --include_path ./tasks \
          --tasks {task} \
          --output {output_dir} \
          --log_samples \
          --num_fewshot 0 \
          --batch_size auto \
          --gen_kwargs temperature=0.0 \
          --apply_chat_template
        """
        run_and_check(ssh, lm_eval_cmd, f"Run lm_eval for {task}")

        # 7. Download results
        sftp = ssh.open_sftp()
        remote_dir = f"/root/{output_dir}"
        local_dir  = os.path.join(local_results_dir, task)
        download_dir(sftp, remote_dir, local_dir)
        sftp.close()

        log("Job completed successfully.")
        return (model, task, True)

    except Exception as e:
        log(f"❌ Error: {e}")
        return (model, task, False)

    finally:
        try:
            runpod.terminate_pod(pod_id)
            log(f"Pod {pod_id} terminated.")
        except Exception as e:
            log(f"⚠ Pod termination failed: {e}")
        if 'ssh' in locals():
            ssh.close()


In [41]:
# Parallel Driver
def run_missing_models_parallel(missing_dict, image_name, gpu_type,
                                yaml_templates, base_output_dir,
                                local_results_dir, log_dir,
                                max_concurrent=3):
    """
    Run all missing model-task pairs with up to `max_concurrent` pods at once.
    Logs each job to its own file and prints live progress grouped by job.
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed
    Path(log_dir).mkdir(parents=True, exist_ok=True)

    futures = []
    with ThreadPoolExecutor(max_workers=max_concurrent) as executor:
        for model, tasks in missing_dict.items():
            for task in tasks:
                futures.append(
                    executor.submit(
                        run_single_model_task,
                        model, task,
                        image_name, gpu_type,
                        yaml_templates,
                        base_output_dir,
                        local_results_dir,
                        log_dir
                    )
                )

        # As each job finishes, print a concise summary
        for fut in as_completed(futures):
            model, task, success = fut.result()
            print(f"[SUMMARY] {model} | {task} → {'✅ success' if success else '❌ failed'}")


In [24]:
# troubleshooting dict if needed
# missing_dict = {
#     "mistral:7b": ["sysengbench-c"],
#     "mistral:instruct": ["sysengbench-c"]
# }

In [ ]:
IMAGE_NAME     = "runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04"
# GPU_TYPE       = "NVIDIA A40"
GPU_TYPE       = "NVIDIA H200"

import os
base_dir = os.getcwd()   # folder where the notebook is running
local_results_dir = os.path.join(base_dir, "output")
log_dir           = os.path.join(base_dir, "runpod_logs")

run_missing_models_parallel(
    missing_dict,
    IMAGE_NAME,
    GPU_TYPE,
    yaml_templates,
    base_output_dir="output",
    local_results_dir=local_results_dir,
    log_dir=log_dir,
    max_concurrent=4   # run 8 pods at the same time
)


# Terminate ALL Pods. BE CAREFUL

## lm-eval pods only

In [1]:
import os
import time
import runpod
import paramiko  # for SSH
from pathlib import Path

# ───────────────────────────────────────────────
# CONFIGURATION
# ───────────────────────────────────────────────
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]  # set in your environment
runpod.api_key = RUNPOD_API_KEY

In [2]:
import runpod

def terminate_lm_eval_pods():
    """
    Find every RunPod pod whose name starts with 'lm-eval-' and terminate it.
    """
    pods = runpod.get_pods()   # list all pods in your account
    if not pods:
        print("No pods found.")
        return

    # Filter only pods created by the evaluation runner
    lm_eval_pods = [p for p in pods if p.get("name", "").startswith("lm-eval-")]
    if not lm_eval_pods:
        print("No lm-eval pods to terminate.")
        return

    print(f"Found {len(lm_eval_pods)} lm-eval pods.")
    for pod in lm_eval_pods:
        pod_id = pod["id"]
        name   = pod["name"]
        status = pod.get("desiredStatus")
        print(f"Terminating {name} ({pod_id}) – current status: {status}")
        try:
            runpod.terminate_pod(pod_id)
            print(f"  ✔ Terminated {name} ({pod_id})")
        except Exception as e:
            print(f"  ❌ Could not terminate {name} ({pod_id}): {e}")

if __name__ == "__main__":
    terminate_lm_eval_pods()


Found 3 lm-eval pods.
Terminating lm-eval-llama4-16x17b-sysengbench-a-1763154138 (1q4nocmy4mf1fa) – current status: RUNNING
  ✔ Terminated lm-eval-llama4-16x17b-sysengbench-a-1763154138 (1q4nocmy4mf1fa)
Terminating lm-eval-llama4-16x17b-sysengbench-1763154138 (dq6zcuglf3pi2w) – current status: RUNNING
  ✔ Terminated lm-eval-llama4-16x17b-sysengbench-1763154138 (dq6zcuglf3pi2w)
Terminating lm-eval-llama3.3-70b-sysengbench-a-1763154138 (wiy49v0bkhv9iy) – current status: RUNNING
  ✔ Terminated lm-eval-llama3.3-70b-sysengbench-a-1763154138 (wiy49v0bkhv9iy)


# ALL PODS

In [ ]:
import runpod

def terminate_all_pods():
    """
    List all pods in your RunPod account and terminate every one.
    Prints each ID and status.
    """
    pods = runpod.get_pods()   # retrieves all pods you own
    if not pods:
        print("No pods found.")
        return

    print(f"Found {len(pods)} pods.")
    for pod in pods:
        pod_id = pod.get("id")
        name   = pod.get("name")
        status = pod.get("desiredStatus")
        print(f"Terminating {name} ({pod_id}) – current status: {status}")
        try:
            runpod.terminate_pod(pod_id)
            print(f"  ✔ Terminated {name} ({pod_id})")
        except Exception as e:
            print(f"  ❌ Could not terminate {name} ({pod_id}): {e}")

if __name__ == "__main__":
    terminate_all_pods()
